In [ ]:
# 加入了超参数
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from dataset_create import prepare_multiple_datasets_and_scaler, ProcessedDrivingDataset, inverse_transform
from DPAI_model import create_model
from datetime import datetime

class AutomaticWeightedLoss(nn.Module):
    """
    基于同方差不确定性的多任务动态 Loss 权重分配
    """
    def __init__(self, num_tasks=2):
        super(AutomaticWeightedLoss, self).__init__()
        # 针对每个任务定义一个可学习的参数 log(sigma^2)，初始为 0
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))

    def forward(self, loss_velocity, loss_steer):
        log_var_vel = self.log_vars[0]
        log_var_steer = self.log_vars[1]

        # 精度 precision = exp(-log(sigma^2)) = 1 / sigma^2
        precision_vel = torch.exp(-log_var_vel)
        precision_steer = torch.exp(-log_var_steer)

        # 动态加权 Loss：精度 * 误差 + 正则化项
        loss = (precision_vel * loss_velocity + log_var_vel) + \
               (precision_steer * loss_steer + log_var_steer)
        return loss

def train():
    # 配置文件路径
    input_files = ['carla_data_collect/20260130_164858/csv/global_vehicle_data.csv',
                'carla_data_collect/20260131_203749/csv/global_vehicle_data.csv',
                 'carla_data_collect/20260131_203911/csv/global_vehicle_data.csv',
                 'carla_data_collect/20260131_204009/csv/global_vehicle_data.csv',
                 'carla_data_collect/20260131_204340/csv/global_vehicle_data.csv']
                   # 'carla_data_collect/20260131_204115/csv/global_vehicle_data.csv',
                   # 'carla_data_collect/20260131_204417/csv/global_vehicle_data.csv',
                   # 'carla_data_collect/20260226_161208/csv/global_vehicle_data.csv']
    output_csv = 'processed_data.csv'
    scaler_json_path = 'scaler_params.json'
    target_names = ['acceleration_x', 'acceleration_y', 'acceleration_z', 'steer']

    # 数据预处理
    print(">>> 开始检查并预处理数据...")
    scaler_params = prepare_multiple_datasets_and_scaler(
        input_files=input_files,
        output_csv=output_csv,
        scaler_json_path=scaler_json_path,
        seq_length=9
    )
    print(">>> 预处理完成！\n")

    # 加载数据集
    dataset = ProcessedDrivingDataset(csv_file=output_csv)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)

    # 初始化模型、损失函数、优化器
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    awl = AutomaticWeightedLoss(num_tasks=2).to(device)
    model = create_model().to(device)
    optimizer = Adam([
        {'params': model.parameters(), 'lr': 1e-4},
        {'params': awl.parameters(), 'lr': 1e-3}
    ])
    scheduler = StepLR(optimizer, step_size=10, gamma=0.9)
    criterion_huber = nn.HuberLoss(delta=0.1)

    # 设置加权损失的权重超参数
    # Loss_name = 'MSELoss'
    Loss_name = 'HuberLoss'
    av = 3000   # 速度权重
    bs = 7000   # 转角权重

    # 日志文件
    log_dir = "logs"
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, f"{Loss_name}_{av}_{bs}.txt")

    # 训练循环
    num_epochs = 25
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        velocity_loss = 0.0
        steer_loss = 0.0

        for batch_idx, (images, state_seq, target) in enumerate(dataloader):
            # 数据移至设备
            img_t_minus_2, img_t_minus_1, img_t = [img.to(device) for img in images]
            state_seq = state_seq.to(device)
            target = target.to(device)

            # 前向传播
            predictions = model(img_t_minus_2, img_t_minus_1, img_t, state_seq)

            # 计算加权损失
            if Loss_name == 'MSELoss':
                velocity_loss = nn.MSELoss()(predictions[:, 0], target[:, 0])  # 速度损失
                steer_loss = nn.MSELoss()(predictions[:, 1], target[:, 1])    # 转角损失
            else:
                velocity_loss = nn.HuberLoss(delta=0.1)(predictions[:, 0], target[:, 0])  # 速度损失
                steer_loss = nn.HuberLoss(delta=0.1)(predictions[:, 1], target[:, 1])    # 转角损失
            # loss = av * velocity_loss + bs * steer_loss
            velocity_loss = criterion_huber(predictions[:, 0], target[:, 0])
            steer_loss = criterion_huber(predictions[:, 1], target[:, 1])
            loss = awl(velocity_loss, steer_loss)
            # 反向传播与优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            velocity_loss += velocity_loss.item()
            steer_loss += steer_loss.item()
            velocity_loss_print = velocity_loss*10000
            steer_loss_print = steer_loss*10000

            # 打印当前 Batch 的损失
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(dataloader)}], "
                  f"Loss: {loss.item():.6f}, vel Loss: {velocity_loss_print:.6f}, ste Loss: {steer_loss_print:.6f}")

        # 学习率调度
        scheduler.step()

        # 记录日志
        with open(log_file, "a") as f:
            f.write(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(dataloader):.6f}\n")
            f.write(f"vel Loss: {velocity_loss_print/len(dataloader):.6f}, ste Loss: {steer_loss_print/len(dataloader):.6f}\n")

        print(f"Epoch [{epoch+1}/{num_epochs}] 完成，平均 Loss: {epoch_loss/len(dataloader):.10f}")

    # 保存模型
    model_path = os.path.join(log_dir, f"{av}_{bs}_{Loss_name}.pth")
    torch.save(model.state_dict(), model_path)
    print(f"模型已保存至 {model_path}")

if __name__ == "__main__":
    train()

>>> 开始检查并预处理数据...
>>> 预处理完成！

Epoch [1/25], Batch [1/518], Loss: 2502.468506, vel Loss: 8715.703125, ste Loss: 3414.608643
Epoch [1/25], Batch [2/518], Loss: 505.572723, vel Loss: 3142.160889, ste Loss: 97.853134
Epoch [1/25], Batch [3/518], Loss: 293.154846, vel Loss: 1624.662109, ste Loss: 141.301514
Epoch [1/25], Batch [4/518], Loss: 394.292694, vel Loss: 1444.964111, ste Loss: 507.280243
Epoch [1/25], Batch [5/518], Loss: 264.231903, vel Loss: 1413.598511, ste Loss: 149.120316
Epoch [1/25], Batch [6/518], Loss: 294.175171, vel Loss: 1661.762939, ste Loss: 128.316360
Epoch [1/25], Batch [7/518], Loss: 389.638489, vel Loss: 1999.563232, ste Loss: 256.297180
Epoch [1/25], Batch [8/518], Loss: 269.316650, vel Loss: 987.137573, ste Loss: 346.417206
Epoch [1/25], Batch [9/518], Loss: 16.664064, vel Loss: 81.836311, ste Loss: 12.538906
Epoch [1/25], Batch [10/518], Loss: 356.190613, vel Loss: 2326.251953, ste Loss: 20.722319


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms
from DPAI_model import create_model

def visualize_attention(model_path, img_t, img_t1, img_t2, state_seq, save_path='attention_map.png'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. 加载模型
    model = create_model().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # 2. 前向传播获取注意力图
    with torch.no_grad():
        # 假设输入已经是预处理好的 Tensor (1, 3, H, W)
        preds, attn_map = model(img_t2, img_t1, img_t, state_seq, return_attn=True)

    # attn_map 形状为 (1, 1, H_feat, W_feat)
    attn_data = attn_map.cpu().squeeze().numpy()

    # 3. 归一化注意力图到 0-255 用于 OpenCV 可视化
    attn_data = (attn_data - attn_data.min()) / (attn_data.max() - attn_data.min())
    attn_data = (attn_data * 255).astype(np.uint8)

    # 4. 上采样到原图大小
    # 假设原图输入大小是 (宽度, 高度)
    orig_img = img_t.cpu().squeeze().permute(1, 2, 0).numpy()
    orig_img = (orig_img * 255).astype(np.uint8)
    h, w, _ = orig_img.shape

    heatmap = cv2.resize(attn_data, (w, h))
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # 5. 叠加显示
    result = cv2.addWeighted(orig_img, 0.6, heatmap, 0.4, 0)

    # 6. 保存或展示
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB))
    plt.subplot(1, 2, 2)
    plt.title("Attention Heatmap")
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    plt.savefig(save_path)
    plt.show()

# 示例调用
if __name__ == "__main__":
    # 这里需要构造你的输入数据，此处仅为结构示例
    model_path = '3000_7000_HuberLoss.pth'
    visualize_attention(model_path, img_t, img_t1, img_t2, state_seq)
    pass